# Lab 7: Kalman Filter

## 1. Step Response — Estimate Drag and Momentum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Collect step response data from robot

Connect to the robot, run PID with a constant PWM step input, and retrieve the logged ToF + motor data.

In [ ]:
import asyncio
import sys
sys.path.insert(0, ".")
from main_python.ble.connection import BLEConnection
from main_python.commands.pid import (
    PIDStart, PIDStop, PIDSetpoint, PIDGains, PIDParams, SendPIDData,
)
from main_python.commands.motors import MotorCmd

In [ ]:
# Collect step response: drive forward at constant PWM and log ToF data
# Adjust STEP_PWM and DURATION_MS for your robot
STEP_PWM = 100  # PWM step input (out of 255)
DURATION_MS = 5000

async def collect_step_response():
    async with BLEConnection() as robot:
        # Drive forward at constant PWM
        await robot.execute(MotorCmd(left=STEP_PWM, right=STEP_PWM))
        await asyncio.sleep(DURATION_MS / 1000)
        await robot.execute(MotorCmd(left=0, right=0))

        # Retrieve logged PID data (time, tof, pwm, etc.)
        pid_data = await robot.execute(SendPIDData())
        return pid_data

# pid_data = await collect_step_response()
# Uncomment above when connected to robot

### Load saved data

Load from a saved CSV/pickle, or populate arrays manually from robot data.

In [ ]:
# Option A: Load from PID data collected above
# times = np.array([s.time for s in pid_data]) / 1000.0  # ms -> s
# tof = np.array([s.measurement for s in pid_data])      # mm
# pwm = np.array([s.pwm for s in pid_data])

# Option B: Load from saved file
# import pickle
# with open("step_response.pkl", "rb") as f:
#     saved = pickle.load(f)
#     times, tof, pwm = saved["times"], saved["tof"], saved["pwm"]

# Placeholder: replace with your data
times = np.array([])  # seconds
tof = np.array([])    # distance in mm
pwm = np.array([])    # PWM values (0-255)

### Plot ToF distance, velocity, and motor input

In [ ]:
# Compute velocity from ToF distance (negative because distance decreases)
dt = np.diff(times)
velocity = -np.diff(tof) / dt  # mm/s

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axes[0].plot(times, tof)
axes[0].set_ylabel("Distance (mm)")
axes[0].set_title("ToF Sensor Output")
axes[0].grid(True)

axes[1].plot(times[1:], velocity)
axes[1].set_ylabel("Velocity (mm/s)")
axes[1].set_title("Computed Velocity")
axes[1].grid(True)

axes[2].plot(times, pwm)
axes[2].set_ylabel("PWM")
axes[2].set_title("Motor Input")
axes[2].set_xlabel("Time (s)")
axes[2].grid(True)

plt.tight_layout()
plt.savefig("../images/step_response.png", dpi=150, bbox_inches="tight")
plt.show()

### Compute drag and momentum

From the step response:
- Steady state speed: the velocity when the car stops accelerating
- 90% rise time: time to reach 90% of steady state speed
- d (drag) = 1 / steady_state_speed
- m (momentum) = -d * t_90 / ln(0.1)

In [ ]:
# Fill in from your step response plots
steady_state_speed = 0  # mm/s — read from velocity plot
t_90 = 0                # seconds — time to reach 90% of steady state speed
u_step = STEP_PWM / 255.0  # normalized step input (0 to 1)

d = 1.0 / steady_state_speed  # drag
m = -d * t_90 / np.log(0.1)   # momentum

print(f"Steady state speed: {steady_state_speed} mm/s")
print(f"90% rise time: {t_90} s")
print(f"Drag (d): {d:.6f}")
print(f"Momentum (m): {m:.6f}")

## 2. Initialize Kalman Filter

Set up A, B, C matrices and covariance matrices.

In [ ]:
# State space model:
#   x = [position, velocity]^T
#   A = [[0, 1], [0, -d/m]]
#   B = [[0], [1/m]]
#   C = [[-1, 0]]  (ToF measures negative position — distance to wall)

A = np.array([[0, 1],
              [0, -d / m]])
B = np.array([[0],
              [1 / m]])
C = np.array([[-1, 0]])

print("A =", A)
print("B =", B)
print("C =", C)

In [ ]:
# Discretize
Delta_T = np.mean(dt)  # average sampling period from your data
n = 2  # state dimension

Ad = np.eye(n) + Delta_T * A
Bd = Delta_T * B

print(f"Delta_T = {Delta_T:.4f} s")
print("Ad =", Ad)
print("Bd =", Bd)

In [ ]:
# Covariance matrices — tune these
# Process noise: how much we trust the model
sigma_1 = 10   # position process noise (mm)
sigma_2 = 10   # velocity process noise (mm/s)

# Sensor noise: how much we trust the ToF sensor
sigma_3 = 20   # measurement noise (mm)

Sigma_u = np.array([[sigma_1**2, 0],
                    [0, sigma_2**2]])
Sigma_z = np.array([[sigma_3**2]])

print("Sigma_u =", Sigma_u)
print("Sigma_z =", Sigma_z)

## 3. Implement and Test Kalman Filter

In [ ]:
def kf(mu, sigma, u, y):
    """Kalman filter update step (follows lecture slide format)."""
    mu_p = Ad.dot(mu) + Bd.dot(u)
    sigma_p = Ad.dot(sigma.dot(Ad.T)) + Sigma_u

    sigma_m = C.dot(sigma_p.dot(C.T)) + Sigma_z
    kkf_gain = sigma_p.dot(C.T.dot(np.linalg.inv(sigma_m)))

    y_m = y - C.dot(mu_p)
    mu = mu_p + kkf_gain.dot(y_m)
    sigma = (np.eye(2) - kkf_gain.dot(C)).dot(sigma_p)

    return mu, sigma

In [ ]:
# Run KF over collected data
# Initialize state with first ToF reading
mu = np.array([[-tof[0]], [0]])
sigma = np.eye(2) * 100  # initial uncertainty

kf_positions = []
kf_velocities = []

for i in range(len(tof)):
    # Normalize PWM to step input fraction
    u = np.array([[pwm[i] / 255.0]])
    y = np.array([[-tof[i]]])

    mu, sigma = kf(mu, sigma, u, y)
    kf_positions.append(-mu[0, 0])  # negate back to distance
    kf_velocities.append(mu[1, 0])

kf_positions = np.array(kf_positions)
kf_velocities = np.array(kf_velocities)

In [ ]:
# Plot KF estimate vs raw ToF
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(times, tof, "b.", markersize=3, label="ToF (raw)")
axes[0].plot(times, kf_positions, "r-", linewidth=1.5, label="KF estimate")
axes[0].set_ylabel("Distance (mm)")
axes[0].set_title("Kalman Filter: Position Estimate vs Raw ToF")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(times[1:], velocity, "b.", markersize=3, label="Finite diff velocity")
axes[1].plot(times, kf_velocities, "r-", linewidth=1.5, label="KF velocity")
axes[1].set_ylabel("Velocity (mm/s)")
axes[1].set_xlabel("Time (s)")
axes[1].set_title("Kalman Filter: Velocity Estimate")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("../images/kalman_filter.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Save data for reuse

In [ ]:
import pickle

data_to_save = {
    "times": times,
    "tof": tof,
    "pwm": pwm,
    "kf_positions": kf_positions,
    "kf_velocities": kf_velocities,
    "d": d,
    "m": m,
    "sigma_1": sigma_1,
    "sigma_2": sigma_2,
    "sigma_3": sigma_3,
}

with open("step_response.pkl", "wb") as f:
    pickle.dump(data_to_save, f)
print("Saved to step_response.pkl")